# Class 7 — Business Model & Metrics

## 7A — Model the Business Workflow

The objective of this stage is to translate the taxi operational
data into a simple business workflow model.

The model identifies the primary operational entity, the operational
event, relevant reference entities, and the measurable outcomes that
support the defined business questions and KPIs.

                    TAXI OPERATION
                         │
                         ▼
                  Trip is initiated
                         │
                         ▼
                  Pickup occurs
                         │
                         ▼
                  Trip is underway
                         │
                         ▼
                  Dropoff occurs
                         │
                         ▼
                Trip is completed
                         │
                         ▼
              Operational metrics

## 7A.4 — Business Entities

The model contains one primary operational entity and one reference
entity directly supported by the retrieved sources.

Entity 1 — Taxi Trip

This is the primary operational entity.

One row represents one trip record.

Important attributes:

Taxi Trip
│
├── pickup timestamp
├── dropoff timestamp
├── pickup location ID
├── dropoff location ID
└── trip distance
Entity 2 — Taxi Zone

This is our reference entity.

It provides:

Taxi Zone
│
├── LocationID
├── Borough
├── Zone
└── service_zone

The relationship:

                  ┌─────────────────┐
                  │    TAXI TRIP    │
                  │                 │
                  │ PULocationID ───┼─────┐
                  │ DOLocationID ───┼──┐  │
                  └─────────────────┘  │  │
                                       │  │
                                       ▼  ▼
                              ┌─────────────────┐
                              │   TAXI ZONE     │
                              │                 │
                              │ LocationID      │
                              │ Borough         │
                              │ Zone            │
                              │ service_zone    │
                              └─────────────────┘

Identify the event

The central event in our model is:

Taxi Trip

                    TAXI TRIP EVENT
                          │
              ┌───────────┼───────────┐
              ▼           ▼           ▼
           START         END       MOVEMENT
              │           │           │
          pickup       dropoff     distance
          time          time        traveled

### State interpretation

The source does not provide an explicit trip-status field.

For this project, trip completion is inferred from the presence of
both pickup and dropoff timestamps. Timestamp chronology is validated
separately.

These are derived interpretations rather than directly observed
status values.

Identify interactions

An interaction is something happening between entities.

For our model:

Taxi Trip
    │
    ├── pickup location → Taxi Zone
    │
    └── dropoff location → Taxi Zone

## Business workflow diagram

                    ┌──────────────────────┐
                    │      Taxi Trip       │
                    │   Operational Event  │
                    └──────────┬───────────┘
                               │
              ┌────────────────┼────────────────┐
              │                │                │
              ▼                ▼                ▼
        Pickup Time       Dropoff Time      Distance
              │                │                │
              └────────────┬───┴────────────────┘
                           ▼
                    Trip Characteristics
                           │
                ┌──────────┼──────────┐
                ▼          ▼          ▼
             Duration   Distance   Locations
                │          │          │
                │          │       ┌──┴──┐
                │          │       ▼     ▼
                │          │    Pickup  Dropoff
                │          │       │     │
                │          │       └──┬──┘
                │          │          ▼
                │          │      Taxi Zone
                │          │
                ▼          ▼
          Business Metrics
                │
      ┌─────────┼─────────┐
      ▼         ▼         ▼
    Volume   Duration   Distance

| Business Question | Data Needed | Entity/Event | Metric |
|---|---|---|---|
| How much taxi activity occurred? | Trip records | Taxi Trip event | Trip Volume |
| How long do trips typically take? | Pickup + dropoff timestamps | Taxi Trip event | Average / Median Duration |
| What distance is typically covered? | trip_distance | Taxi Trip event | Average Trip Distance |
| Which pickup locations have highest activity? | PULocationID + Zone | Trip + Taxi Zone | Pickup Trip Volume |
| What proportion of trips have invalid duration? | Pickup + dropoff timestamps | Taxi Trip event | Invalid Trip Duration Rate |
| What proportion have location issues? | LocationIDs + zone reference | Trip + Taxi Zone | Invalid/Unmatched Location Rate |

## Model Boundaries

The current model represents trip-level operational activity and
taxi-zone reference information.

It does not represent:

- individual driver identity
- individual passenger identity
- vehicle identity
- dispatch decisions
- traffic conditions
- weather
- customer cancellations
- causal reasons for long or short trips
- operational interventions and their outcomes

These factors are outside the observable scope of the selected
sources.


## Final business workflow model
                                                                       TAXI OPERATION
                              │
                              ▼
                      ┌───────────────┐
                      │   Taxi Trip   │
                      │     Event     │
                      └───────┬───────┘
                              │
          ┌───────────────────┼───────────────────┐
          │                   │                   │
          ▼                   ▼                   ▼
     Pickup Time         Dropoff Time        Distance
          │                   │                   │
          └─────────────┬─────┴───────────────────┘
                        ▼
                  Trip Duration
                        │
            ┌───────────┴───────────┐
            ▼                       ▼
       Operational              Data Quality
        Metrics                    Metrics
            │                       │
    ┌───────┼────────┐       ┌──────┴───────┐
    ▼       ▼        ▼       ▼              ▼
  Volume Duration Distance Invalid       Location
                    │       Duration       Rate
                    │
                    ▼
             Pickup / Dropoff
                  Locations
                    │
                    ▼
                Taxi Zone

## lass 7 — Step 7B: Design the Relational/Event Data Model

                 ┌─────────────────────┐
                 │      dim_zone       │
                 │─────────────────────│
                 │ LocationID (PK)     │
                 │ Borough             │
                 │ Zone                │
                 │ service_zone        │
                 └──────────┬──────────┘
                            ▲
                            │
                  ┌─────────┴─────────┐
                  │                   │
          pickup_location_id   dropoff_location_id
                  │                   │
                  └─────────┬─────────┘
                            │
                 ┌──────────▼──────────┐
                 │      fact_trip      │
                 │─────────────────────│
                 │ trip_id             │
                 │ pickup_datetime     │
                 │ dropoff_datetime    │
                 │ pickup_location_id  │
                 │ dropoff_location_id │
                 │ trip_distance       │
                 │ trip_duration_min   │
                 │ is_valid            │
                 └─────────────────────┘

## 7B.2 — Fact Trip

`fact_trip` represents the primary operational event in the model.

### Grain

One row represents one taxi trip record.

### Purpose

The table contains the timestamps, locations, distance, derived
duration, and validation status required to calculate the project's
operational and data-quality metrics.

| Column                  | Meaning                                 | Type     | Role         |
| ----------------------- | --------------------------------------- | -------- | ------------ |
| `trip_id`               | Unique identifier assigned by our model | integer  | Primary key  |
| `pickup_datetime`       | Trip pickup timestamp                   | datetime | Event start  |
| `dropoff_datetime`      | Trip dropoff timestamp                  | datetime | Event end    |
| `pickup_location_id`    | Pickup zone reference                   | integer  | Foreign key  |
| `dropoff_location_id`   | Dropoff zone reference                  | integer  | Foreign key  |
| `trip_distance`         | Recorded trip distance                  | numeric  | Measurement  |
| `trip_duration_minutes` | Derived trip duration                   | numeric  | Measurement  |
| `is_valid_duration`     | Whether chronology is valid             | boolean  | Quality flag |
| `is_valid_location`     | Whether both locations resolve          | boolean  | Quality flag |
| `is_valid_trip`         | Whether core validation passes          | boolean  | Quality flag |


## 7B.4 — Zone Dimension

`dim_zone` represents the taxi-zone reference data used to resolve
pickup and dropoff LocationIDs into geographic attributes.

### Grain

One row represents one taxi zone reference record.

### Primary key

`LocationID`

## Evenet model

                 FACT TRIP EVENT
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ▼              ▼              ▼
     START            END         MEASUREMENT
        │              │              │
 pickup_datetime  dropoff_datetime  distance
        │              │              │
        └──────────────┼──────────────┘
                       ▼
                trip_duration

## Map model to our KPIs
# Business KPIs
| KPI                       | Source table | Fields                  |
| ------------------------- | ------------ | ----------------------- |
| **Trip Volume**           | `fact_trip`  | `trip_id`               |
| **Average Trip Duration** | `fact_trip`  | `trip_duration_minutes` |
| **Median Trip Duration**  | `fact_trip`  | `trip_duration_minutes` |
| **Average Trip Distance** | `fact_trip`  | `trip_distance`         |

# Additional operational breakdown
| Analysis              | Tables                   | Fields                                  |
| --------------------- | ------------------------ | --------------------------------------- |
| Pickup volume by zone | `fact_trip` + `dim_zone` | `pickup_location_id`, `Zone`, `Borough` |

# Data-quality KPIs
| KPI                                 | Source                   | Rule                        |
| ----------------------------------- | ------------------------ | --------------------------- |
| **Invalid Trip Duration Rate**      | `fact_trip`              | `is_valid_duration = False` |
| **Invalid/Unmatched Location Rate** | `fact_trip` + `dim_zone` | `is_valid_location = False` |


## Important denominator decision

This is something we should make explicit now.

For Trip Volume:

All July reporting trip records

For Average/Median Duration:

Trips with valid duration

For Average Distance:

Trips with valid/non-missing distance

For Invalid Duration Rate:

Invalid duration trips
────────────────────────────
Trips with required timestamps

For Invalid Location Rate:

Location-issue trips
─────────────────────
All July reporting trips

These denominator decisions need to be documented because otherwise two analysts could calculate the same KPI differently.

## Model Diagram
                         ┌────────────────────────┐
                         │       dim_zone         │
                         │────────────────────────│
                         │ LocationID       PK    │
                         │ Borough                │
                         │ Zone                   │
                         │ service_zone           │
                         └───────────▲────────────┘
                                     │
                    ┌────────────────┴────────────────┐
                    │                                 │
             pickup_location_id              dropoff_location_id
                    │                                 │
                    └────────────────┬────────────────┘
                                     │
                         ┌───────────▼────────────┐
                         │       fact_trip        │
                         │────────────────────────│
                         │ trip_id          PK    │
                         │ pickup_datetime        │
                         │ dropoff_datetime       │
                         │ pickup_location_id FK  │
                         │ dropoff_location_id FK │
                         │ trip_distance          │
                         │ trip_duration_minutes  │
                         │ is_valid_duration      │
                         │ is_valid_location      │
                         │ is_valid_trip          │
                         └───────────┬────────────┘
                                     │
                                     ▼
                              KPI / Analysis
                                     │
                  ┌──────────────────┼─────────────────┐
                  ▼                  ▼                 ▼
             Trip Volume        Duration          Distance
                                Avg / Median          Avg

## 7B.14 — Model Boundaries

The modeled dataset represents trip-level operational events and
taxi-zone reference information.

The model does not introduce entities that are not supported by the
available source data.

Therefore, the model does not contain:

- Driver
- Passenger
- Vehicle
- Dispatch event
- Customer
- Traffic condition
- Weather
- Intervention

The `trip_id` used by the modeled fact table is a model-generated
technical identifier and is not claimed to be a source-system trip ID.

## 7C.1 — Load Source Data

The modeled dataset is built from the preserved raw trip and zone
reference sources.

The July 2026 reporting period is defined using pickup timestamp.

In [1]:
import pandas as pd

trip_path = "../data/raw/trips/yellow_tripdata_2026-07.parquet"
zone_path = "../data/raw/zones/taxi_zone_lookup.csv"

trips = pd.read_parquet(trip_path)
zones = pd.read_csv(zone_path)

july_2026_trips = trips[
    (trips["tpep_pickup_datetime"] >= "2026-07-01") &
    (trips["tpep_pickup_datetime"] < "2026-08-01")
].copy()

print("Raw trips:", len(trips))
print("July trips:", len(july_2026_trips))
print("Zone rows:", len(zones))

Raw trips: 3530109
July trips: 3530063
Zone rows: 265


## 7C.2 — Build `dim_zone`

`dim_zone` represents the taxi-zone reference entity.

Its grain is one row per `LocationID`.

In [2]:
dim_zone = zones[
    [
        "LocationID",
        "Borough",
        "Zone",
        "service_zone"
    ]
].copy()

In [3]:
print("dim_zone rows:", len(dim_zone))
print(
    "Unique LocationIDs:",
    dim_zone["LocationID"].nunique()
)
print(
    "Duplicate LocationIDs:",
    dim_zone["LocationID"].duplicated().sum()
)

dim_zone rows: 265
Unique LocationIDs: 265
Duplicate LocationIDs: 0


In [4]:
fact_trip = july_2026_trips[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ]
].copy()

In [5]:
fact_trip = fact_trip.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id"
})

In [6]:
print(fact_trip.head())
print("\nRows:", len(fact_trip))
print("Columns:", fact_trip.columns.tolist())

      pickup_datetime    dropoff_datetime  pickup_location_id  \
0 2026-07-01 00:36:25 2026-07-01 00:43:18                 138   
1 2026-07-01 00:02:53 2026-07-01 00:17:12                 162   
2 2026-07-01 00:18:56 2026-07-01 00:27:38                 137   
3 2026-07-01 00:35:06 2026-07-01 00:57:51                 138   
4 2026-07-01 00:36:59 2026-07-01 00:36:59                 140   

   dropoff_location_id  trip_distance  
0                  138           2.00  
1                  112           3.20  
2                  263           3.18  
3                  246          10.00  
4                  106           9.62  

Rows: 3530063
Columns: ['pickup_datetime', 'dropoff_datetime', 'pickup_location_id', 'dropoff_location_id', 'trip_distance']


In [7]:
fact_trip.insert(
    0,
    "trip_id",
    range(1, len(fact_trip) + 1)
)

In [8]:
print(fact_trip.head())
print(
    "Unique trip IDs:",
    fact_trip["trip_id"].nunique()
)

   trip_id     pickup_datetime    dropoff_datetime  pickup_location_id  \
0        1 2026-07-01 00:36:25 2026-07-01 00:43:18                 138   
1        2 2026-07-01 00:02:53 2026-07-01 00:17:12                 162   
2        3 2026-07-01 00:18:56 2026-07-01 00:27:38                 137   
3        4 2026-07-01 00:35:06 2026-07-01 00:57:51                 138   
4        5 2026-07-01 00:36:59 2026-07-01 00:36:59                 140   

   dropoff_location_id  trip_distance  
0                  138           2.00  
1                  112           3.20  
2                  263           3.18  
3                  246          10.00  
4                  106           9.62  
Unique trip IDs: 3530063


In [9]:
print(
    "Trip ID duplicates:",
    fact_trip["trip_id"].duplicated().sum()
)

Trip ID duplicates: 0


In [10]:
fact_trip["trip_duration_minutes"] = (
    fact_trip["dropoff_datetime"]
    - fact_trip["pickup_datetime"]
).dt.total_seconds() / 60

In [11]:
print(
    fact_trip["trip_duration_minutes"].describe()
)

count    3.530063e+06
mean     1.729142e+01
std      2.707407e+01
min     -1.666667e-01
25%      8.483333e+00
50%      1.416667e+01
75%      2.216667e+01
max      1.716542e+04
Name: trip_duration_minutes, dtype: float64


In [12]:
fact_trip["is_valid_duration"] = (
    fact_trip["pickup_datetime"].notna()
    &
    fact_trip["dropoff_datetime"].notna()
    &
    (
        fact_trip["dropoff_datetime"]
        >=
        fact_trip["pickup_datetime"]
    )
)

In [13]:
print(
    fact_trip["is_valid_duration"].value_counts()
)

is_valid_duration
True     3530062
False          1
Name: count, dtype: int64


In [14]:
zone_ids = set(
    dim_zone["LocationID"]
)

In [15]:
pickup_location_valid = (
    fact_trip["pickup_location_id"]
    .isin(zone_ids)
)

dropoff_location_valid = (
    fact_trip["dropoff_location_id"]
    .isin(zone_ids)
)

fact_trip["is_valid_location"] = (
    pickup_location_valid
    &
    dropoff_location_valid
)


In [16]:
print(
    fact_trip["is_valid_location"].value_counts()
)

is_valid_location
True    3530063
Name: count, dtype: int64


In [17]:
fact_trip["is_valid_distance"] = (
    fact_trip["trip_distance"].notna()
    &
    (fact_trip["trip_distance"] >= 0)
)

In [18]:
print(
    fact_trip["is_valid_distance"].value_counts()
)

is_valid_distance
True    3530063
Name: count, dtype: int64


In [19]:
fact_trip["is_valid_trip"] = (
    fact_trip["is_valid_duration"]
    &
    fact_trip["is_valid_location"]
    &
    fact_trip["is_valid_distance"]
)

In [20]:
print(
    fact_trip["is_valid_trip"].value_counts()
)

is_valid_trip
True     3530062
False          1
Name: count, dtype: int64


In [21]:
print("July reporting rows:", len(july_2026_trips))
print("fact_trip rows:", len(fact_trip))

print(
    "Row count preserved:",
    len(july_2026_trips) == len(fact_trip)
)

July reporting rows: 3530063
fact_trip rows: 3530063
Row count preserved: True


In [22]:
print(
    "Unique trip IDs:",
    fact_trip["trip_id"].nunique()
)

print(
    "Duplicate trip IDs:",
    fact_trip["trip_id"].duplicated().sum()
)

Unique trip IDs: 3530063
Duplicate trip IDs: 0


In [23]:
pickup_unmatched = (
    ~fact_trip["pickup_location_id"].isin(
        dim_zone["LocationID"]
    )
).sum()

dropoff_unmatched = (
    ~fact_trip["dropoff_location_id"].isin(
        dim_zone["LocationID"]
    )
).sum()

print("Unmatched pickup references:", pickup_unmatched)
print("Unmatched dropoff references:", dropoff_unmatched)

Unmatched pickup references: 0
Unmatched dropoff references: 0


In [24]:
print("========== FACT TRIP ==========")
print(fact_trip.info())

print("\n========== DIM ZONE ==========")
print(dim_zone.info())

========== FACT TRIP ==========
<class 'pandas.DataFrame'>
Index: 3530063 entries, 0 to 3530108
Data columns (total 11 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   trip_id                int64         
 1   pickup_datetime        datetime64[us]
 2   dropoff_datetime       datetime64[us]
 3   pickup_location_id     int32         
 4   dropoff_location_id    int32         
 5   trip_distance          float64       
 6   trip_duration_minutes  float64       
 7   is_valid_duration      bool          
 8   is_valid_location      bool          
 9   is_valid_distance      bool          
 10  is_valid_trip          bool          
dtypes: bool(4), datetime64[us](2), float64(2), int32(2), int64(1)
memory usage: 202.0 MB
None

========== DIM ZONE ==========
<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Locat

In [25]:
print(
    fact_trip.head()
)

   trip_id     pickup_datetime    dropoff_datetime  pickup_location_id  \
0        1 2026-07-01 00:36:25 2026-07-01 00:43:18                 138   
1        2 2026-07-01 00:02:53 2026-07-01 00:17:12                 162   
2        3 2026-07-01 00:18:56 2026-07-01 00:27:38                 137   
3        4 2026-07-01 00:35:06 2026-07-01 00:57:51                 138   
4        5 2026-07-01 00:36:59 2026-07-01 00:36:59                 140   

   dropoff_location_id  trip_distance  trip_duration_minutes  \
0                  138           2.00               6.883333   
1                  112           3.20              14.316667   
2                  263           3.18               8.700000   
3                  246          10.00              22.750000   
4                  106           9.62               0.000000   

   is_valid_duration  is_valid_location  is_valid_distance  is_valid_trip  
0               True               True               True           True  
1               Tr

In [26]:
model_summary = {
    "fact_trip_rows": len(fact_trip),
    "fact_trip_unique_ids": fact_trip["trip_id"].nunique(),
    "dim_zone_rows": len(dim_zone),
    "invalid_duration": (~fact_trip["is_valid_duration"]).sum(),
    "invalid_location": (~fact_trip["is_valid_location"]).sum(),
    "invalid_distance": (~fact_trip["is_valid_distance"]).sum(),
    "invalid_trip": (~fact_trip["is_valid_trip"]).sum()
}

for key, value in model_summary.items():
    print(f"{key}: {value}")

fact_trip_rows: 3530063
fact_trip_unique_ids: 3530063
dim_zone_rows: 265
invalid_duration: 1
invalid_location: 0
invalid_distance: 0
invalid_trip: 1


## 7C.15 — Transformation Summary

The modeled dataset is created from the July 2026 reporting subset.

### `fact_trip`

The trip source fields are renamed to business-oriented model names.

A model-generated `trip_id` is assigned to each reporting record.

`trip_duration_minutes` is derived from the pickup and dropoff
timestamps.

Validation flags are created for:

- duration chronology
- location reference integrity
- distance validity
- overall trip validity

### `dim_zone`

The taxi zone reference is modeled at one row per `LocationID`.

No duplicate zone keys were observed.

### Grain preservation

The fact table preserves the original July reporting row count.

No records are silently removed during model construction.
Invalid records remain represented through validation flags.

## Step 7D — Business Metrics

In [28]:
print("fact_trip rows:", len(fact_trip))
print("valid duration rows:", fact_trip["is_valid_duration"].sum())
print("valid distance rows:", fact_trip["is_valid_distance"].sum())
print("valid trip rows:", fact_trip["is_valid_trip"].sum())

fact_trip rows: 3530063
valid duration rows: 3530062
valid distance rows: 3530063
valid trip rows: 3530062


In [29]:
trip_volume = fact_trip["trip_id"].count()

print("Trip Volume:", trip_volume)

Trip Volume: 3530063


In [31]:
valid_duration_trips = fact_trip[
    fact_trip["is_valid_duration"]
]

average_trip_duration = (
    valid_duration_trips["trip_duration_minutes"]
    .mean()
)

print("Average Trip Duration:", average_trip_duration)

Average Trip Duration: 17.291424810669046


In [32]:
print(
    "Average Trip Duration:",
    round(average_trip_duration, 2),
    "minutes"
)

Average Trip Duration: 17.29 minutes


### Average Trip Duration

Average trip duration is calculated using only trips that satisfy
the duration validation rule:

dropoff_datetime >= pickup_datetime

The single trip failing the chronology rule is excluded from the
duration metric but remains in the modeled dataset with its validation
flag preserved.

This prevents an invalid duration from influencing the operational
duration metric without silently deleting the underlying record.

In [33]:
median_trip_duration = (
    valid_duration_trips["trip_duration_minutes"]
    .median()
)

print(
    "Median Trip Duration:",
    round(median_trip_duration, 2),
    "minutes"
)

Median Trip Duration: 14.17 minutes


In [34]:
valid_distance_trips = fact_trip[
    fact_trip["is_valid_distance"]
]

average_trip_distance = (
    valid_distance_trips["trip_distance"]
    .mean()
)

print(
    "Average Trip Distance:",
    round(average_trip_distance, 2),
    "miles"
)

Average Trip Distance: 5.55 miles


In [35]:
business_metrics = pd.DataFrame([
    {
        "metric": "Trip Volume",
        "value": trip_volume,
        "unit": "trips",
        "population": "All July 2026 trip records"
    },
    {
        "metric": "Average Trip Duration",
        "value": round(average_trip_duration, 2),
        "unit": "minutes",
        "population": "Trips with valid duration"
    },
    {
        "metric": "Median Trip Duration",
        "value": round(median_trip_duration, 2),
        "unit": "minutes",
        "population": "Trips with valid duration"
    },
    {
        "metric": "Average Trip Distance",
        "value": round(average_trip_distance, 2),
        "unit": "miles",
        "population": "Trips with valid distance"
    }
])

business_metrics

,metric,value,unit,population
0,Trip Volume,3530063.00,trips,All July 2026 trip records
1,Average Trip Duration,17.29,minutes,Trips with valid duration
2,Median Trip Duration,14.17,minutes,Trips with valid duration
3,Average Trip Distance,5.55,miles,Trips with valid distance


## Business Interpretation

### Trip Volume
Measures the observed taxi trip activity during the July 2026
reporting period.

### Average Trip Duration
Measures the mean duration of trips with valid pickup/dropoff
chronology.

### Median Trip Duration
Measures the middle trip duration among trips with valid duration,
providing a complementary view to the average.

### Average Trip Distance
Measures the average recorded trip distance among trips satisfying
the distance validation rule.

The metrics are calculated from the modeled dataset rather than
directly from the raw source. This makes the metric population and
validation treatment explicit.

## Metric Definitions

| Metric | Numerator / Calculation | Denominator / Population | Validation Treatment |
|---|---|---|---|
| Trip Volume | Count of trip_id | All July trip records | No records excluded |
| Average Trip Duration | Sum of valid trip durations | Valid-duration trips | Invalid chronology excluded |
| Median Trip Duration | Median of valid trip durations | Valid-duration trips | Invalid chronology excluded |
| Average Trip Distance | Sum of valid trip distances | Valid-distance trips | Invalid/null/negative distance excluded |

In [36]:
pickup_zone_volume = (
    fact_trip
    .merge(
        dim_zone,
        left_on="pickup_location_id",
        right_on="LocationID",
        how="left"
    )
    .groupby(["LocationID", "Borough", "Zone"])
    .size()
    .reset_index(name="trip_volume")
    .sort_values("trip_volume", ascending=False)
)

In [37]:
pickup_zone_volume.head(10)

,LocationID,Borough,Zone,trip_volume
155,161,Manhattan,Midtown Center,152733
126,132,Queens,JFK Airport,151816
231,237,Manhattan,Upper East Side South,140423
230,236,Manhattan,Upper East Side North,118463
180,186,Manhattan,Penn Station/Madison Sq West,117790
156,162,Manhattan,Midtown East,113562
224,230,Manhattan,Times Sq/Theatre District,108789
164,170,Manhattan,Murray Hill,97766
136,142,Manhattan,Lincoln Square East,97599
67,68,Manhattan,East Chelsea,96828


In [38]:
kpi_evidence = pd.DataFrame([
    {
        "kpi": "Trip Volume",
        "value": trip_volume,
        "unit": "trips",
        "business_question": "How much taxi activity occurred?",
        "population": "All July 2026 trips"
    },
    {
        "kpi": "Average Trip Duration",
        "value": round(average_trip_duration, 2),
        "unit": "minutes",
        "business_question": "How long do trips typically take?",
        "population": "Valid-duration trips"
    },
    {
        "kpi": "Median Trip Duration",
        "value": round(median_trip_duration, 2),
        "unit": "minutes",
        "business_question": "What is the typical trip duration?",
        "population": "Valid-duration trips"
    },
    {
        "kpi": "Average Trip Distance",
        "value": round(average_trip_distance, 2),
        "unit": "miles",
        "business_question": "What distance is typically covered?",
        "population": "Valid-distance trips"
    }
])

kpi_evidence

,kpi,value,unit,business_question,population
0,Trip Volume,3530063.00,trips,How much taxi activity occurred?,All July 2026 trips
1,Average Trip Duration,17.29,minutes,How long do trips typically take?,Valid-duration trips
2,Median Trip Duration,14.17,minutes,What is the typical trip duration?,Valid-duration trips
3,Average Trip Distance,5.55,miles,What distance is typically covered?,Valid-distance trips


## What we have achieved

                 RAW DATA
                    │
                    ▼
          ┌───────────────────┐
          │   fact_trip       │
          │                   │
          │ trip_id           │
          │ pickup_datetime   │
          │ dropoff_datetime  │
          │ locations         │
          │ distance          │
          │ duration          │
          │ validation flags  │
          └─────────┬─────────┘
                    │
                    │
          ┌─────────▼─────────┐
          │    dim_zone       │
          │                   │
          │ LocationID        │
          │ Borough           │
          │ Zone              │
          │ service_zone      │
          └─────────┬─────────┘
                    │
                    ▼
             BUSINESS METRICS
                    │
       ┌────────────┼─────────────┐
       ▼            ▼             ▼
 Trip Volume   Duration       Distance
                  │
            ┌─────┴─────┐
            ▼           ▼
         Average      Median

## Business Metric Results

The July 2026 reporting dataset contains 3,530,063 taxi trip records.

The resulting business metrics are:

| KPI | Result | Metric Population |
|---|---:|---|
| Trip Volume | 3,530,063 trips | All July 2026 trip records |
| Average Trip Duration | 17.29 minutes | 3,530,062 valid-duration trips |
| Median Trip Duration | 14.17 minutes | 3,530,062 valid-duration trips |
| Average Trip Distance | 5.55 miles | 3,530,063 valid-distance trips |

Duration-based metrics exclude the single record that fails the
pickup/dropoff chronology validation rule. The invalid record is
retained in the modeled dataset and represented through validation
flags rather than silently removed.

The lower median duration compared with the average indicates that
the distribution contains trips with durations above the mean.
This metric difference describes the observed data but does not
establish the cause of longer trips.

## Step 7E — Data Quality KPIs

In [40]:
invalid_duration_count = (
    (~fact_trip["is_valid_duration"]).sum()
)

duration_denominator = (
    fact_trip["pickup_datetime"].notna()
    & fact_trip["dropoff_datetime"].notna()
).sum()

invalid_duration_rate = (
    invalid_duration_count / duration_denominator * 100
)

print("Invalid Duration Count:", invalid_duration_count)
print("Duration Denominator:", duration_denominator)
print(
    "Invalid Trip Duration Rate:",
    round(invalid_duration_rate, 6),
    "%"
)

Invalid Duration Count: 1
Duration Denominator: 3530063
Invalid Trip Duration Rate: 2.8e-05 %


In [41]:
pickup_location_valid = fact_trip["pickup_location_id"].isin(
    dim_zone["LocationID"]
)

dropoff_location_valid = fact_trip["dropoff_location_id"].isin(
    dim_zone["LocationID"]
)

is_valid_location = (
    pickup_location_valid
    & dropoff_location_valid
)

invalid_location_count = (
    (~is_valid_location).sum()
)

location_denominator = len(fact_trip)

invalid_location_rate = (
    invalid_location_count / location_denominator * 100
)

print("Invalid/Unmatched Location Count:", invalid_location_count)
print("Location Denominator:", location_denominator)
print(
    "Invalid/Unmatched Location Rate:",
    round(invalid_location_rate, 6),
    "%"
)

Invalid/Unmatched Location Count: 0
Location Denominator: 3530063
Invalid/Unmatched Location Rate: 0.0 %


In [42]:
print(
    "Model invalid location count:",
    (~fact_trip["is_valid_location"]).sum()
)

print(
    "Recalculated invalid location count:",
    invalid_location_count
)

Model invalid location count: 0
Recalculated invalid location count: 0


In [43]:
all_kpi_evidence = pd.DataFrame([
    {
        "kpi": "Trip Volume",
        "value": trip_volume,
        "unit": "trips",
        "population": "All July 2026 trip records",
        "type": "Business"
    },
    {
        "kpi": "Average Trip Duration",
        "value": round(average_trip_duration, 2),
        "unit": "minutes",
        "population": "Valid-duration trips",
        "type": "Business"
    },
    {
        "kpi": "Median Trip Duration",
        "value": round(median_trip_duration, 2),
        "unit": "minutes",
        "population": "Valid-duration trips",
        "type": "Business"
    },
    {
        "kpi": "Average Trip Distance",
        "value": round(average_trip_distance, 2),
        "unit": "miles",
        "population": "Valid-distance trips",
        "type": "Business"
    },
    {
        "kpi": "Invalid Trip Duration Rate",
        "value": round(invalid_duration_rate, 6),
        "unit": "%",
        "population": "Trips with both timestamps present",
        "type": "Data Quality"
    },
    {
        "kpi": "Invalid/Unmatched Location Rate",
        "value": round(invalid_location_rate, 6),
        "unit": "%",
        "population": "All July 2026 trip records",
        "type": "Data Quality"
    }
])

all_kpi_evidence

,kpi,value,unit,population,type
0,Trip Volume,3.530063e+06,trips,All July 2026 trip records,Business
1,Average Trip Duration,1.729000e+01,minutes,Valid-duration trips,Business
2,Median Trip Duration,1.417000e+01,minutes,Valid-duration trips,Business
3,Average Trip Distance,5.550000e+00,miles,Valid-distance trips,Business
4,Invalid Trip Duration Rate,2.800000e-05,%,Trips with both timestamps present,Data Quality
5,Invalid/Unmatched Location Rate,0.000000e+00,%,All July 2026 trip records,Data Quality


## Known / Unknown / Assumption / Limitation

| Category       | Evidence                                                                                                                           |
| -------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| **Known**      | July 2026 reporting dataset contains 3,530,063 trip records.                                                                       |
| **Known**      | 3,530,062 trips have valid pickup/dropoff chronology.                                                                              |
| **Known**      | 1 trip has invalid duration chronology.                                                                                            |
| **Known**      | No pickup or dropoff LocationID is unmatched against the zone lookup.                                                              |
| **Known**      | Average trip duration among valid-duration trips is 17.29 minutes.                                                                 |
| **Known**      | Median trip duration among valid-duration trips is 14.17 minutes.                                                                  |
| **Known**      | Average recorded trip distance is 5.55 miles.                                                                                      |
| **Assumption** | The reporting period is defined using pickup datetime.                                                                             |
| **Assumption** | A LocationID present in the TLC zone lookup is treated as a valid reference.                                                       |
| **Assumption** | A non-negative recorded trip distance is considered valid for this metric.                                                         |
| **Unknown**    | The dataset does not explain why an individual trip is unusually long or short.                                                    |
| **Unknown**    | The dataset does not provide causal information such as traffic, weather, or road conditions.                                      |
| **Unknown**    | The dataset does not identify the driver or passenger associated with a trip.                                                      |
| **Limitation** | The analysis uses July 2026 only and does not establish trends across other months.                                                |
| **Limitation** | The zone lookup validates LocationID references but does not independently verify the real-world accuracy of each trip's location. |
| **Limitation** | Extreme but technically valid duration/distance values are not automatically classified as invalid.                                |


In [44]:
invalid_duration_records = fact_trip[
    ~fact_trip["is_valid_duration"]
].copy()

invalid_duration_records[
    [
        "trip_id",
        "pickup_datetime",
        "dropoff_datetime",
        "pickup_location_id",
        "dropoff_location_id",
        "trip_distance",
        "trip_duration_minutes",
        "is_valid_duration",
        "is_valid_trip"
    ]
]

,trip_id,pickup_datetime,dropoff_datetime,pickup_location_id,dropoff_location_id,trip_distance,trip_duration_minutes,is_valid_duration,is_valid_trip
3344248,3344203,2026-07-25 22:24:27,2026-07-25 22:24:17,148,148,3.65,-0.166667,False,False


In [47]:
invalid_duration_records[
    [
        "trip_id",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_distance",
        "trip_duration_minutes",
        "is_valid_duration",
        "is_valid_trip"
    ]
]

,trip_id,pickup_datetime,dropoff_datetime,trip_distance,trip_duration_minutes,is_valid_duration,is_valid_trip
3344248,3344203,2026-07-25 22:24:27,2026-07-25 22:24:17,3.65,-0.166667,False,False


### Validation Finding — Invalid Trip Duration

One trip record fails the chronology validation rule.

- trip_id: 3,344,248
- pickup_datetime: 2026-07-25 22:24:27
- dropoff_datetime: 2026-07-25 22:24:17
- trip_distance: 3.65 miles
- calculated duration: -0.166667 minutes

The dropoff timestamp occurs 10 seconds before the pickup timestamp,
which produces a negative trip duration.

The record is retained in the modeled dataset and marked with:

- is_valid_duration = False
- is_valid_trip = False

It is excluded from duration-based business metrics but is not
silently deleted from the dataset.

# Class 7 — Final Decision Summary

## Business Metrics

- Trip Volume: 3,530,063 trips
- Average Trip Duration: 17.29 minutes
- Median Trip Duration: 14.17 minutes
- Average Trip Distance: 5.55 miles

## Data Quality Metrics

- Invalid Trip Duration Rate: 0.00002833%
- Invalid/Unmatched Location Rate: 0%

## Important Validation Finding

One trip record has a dropoff timestamp 10 seconds before its
pickup timestamp, resulting in a negative duration of -0.166667
minutes.

The record is preserved and flagged as invalid rather than silently
corrected or deleted. It is excluded from duration-based KPIs.

## Modeling Boundary

The dataset supports measurement of observed taxi trip activity,
duration, distance, and taxi-zone activity.

It does not establish causal explanations for trip duration or
distance because information such as traffic, weather, driver
identity, passenger identity, and other contextual factors is not
represented in the modeled dataset.